<a href="https://colab.research.google.com/github/GULLOJUUDAYKIRAN5/2203a51238-DAA/blob/main/.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install librosa soundfile

In [ ]:
import os
import random
import shutil
import librosa
import soundfile as sf
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
dataset1_path = "/content/drive/MyDrive/baby_cry/dataset1/cry"
dataset2_path = "/content/drive/MyDrive/baby_cry/dataset1/donateacry_corpus_cleaned_and_updated_data"
augmented1_path = "/content/Augmented_Dataset1"
augmented2_path = "/content/Augmented_Dataset2"
final_path = "/content/Merged_Final"
labels_csv_path = "/content/labels.csv"

os.makedirs(augmented1_path, exist_ok=True)
os.makedirs(augmented2_path, exist_ok=True)
os.makedirs(final_path, exist_ok=True)

In [ ]:
def augment_audio(file_path, save_dir, num_aug=3):
    try:
        y, sr = librosa.load(file_path, sr=None)
        y = librosa.to_mono(y)  # Ensure mono
    except Exception as e:
        print(f"⚠️ Skipped {file_path} due to load error: {e}")
        return

    if len(y) < sr:  # skip very short audio (<1 sec)
        print(f"⚠️ Skipped {file_path} because audio is too short")
        return

    base_name = os.path.splitext(os.path.basename(file_path))[0]

    # Save original
    sf.write(os.path.join(save_dir, f"{base_name}_orig.wav"), y, sr)

    for i in range(num_aug):
        y_aug = y.copy()
        choice = random.choice(["noise", "stretch", "pitch"])
        try:
            if choice == "noise":
                y_aug = y_aug + 0.005 * np.random.randn(len(y_aug))
            elif choice == "stretch":
                rate = random.uniform(0.8, 1.2)
                y_aug = librosa.effects.time_stretch(y_aug, rate=rate)
            elif choice == "pitch":
                steps = random.randint(-2, 2)
                y_aug = librosa.effects.pitch_shift(y_aug, sr=sr, n_steps=steps)
        except Exception as e:
            print(f"⚠️ Skipped augmentation {choice} for {file_path} due to error: {e}")
            continue

        sf.write(os.path.join(save_dir, f"{base_name}_aug{i}.wav"), y_aug, sr)

In [ ]:
def get_all_audio_files(path):
    audio_files = []
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.lower().endswith(".wav"):
                audio_files.append(os.path.join(root, f))
    return audio_files

In [ ]:
def process_dataset(input_path, output_path):
    all_files = get_all_audio_files(input_path)
    if len(all_files) == 0:
        print(f"⚠️ Skipping {input_path} — no .wav files found.")
        return

    for fpath in tqdm(all_files, desc=f"Processing {os.path.basename(input_path)}"):
        augment_audio(fpath, output_path)

In [ ]:
process_dataset(dataset1_path, augmented1_path)
process_dataset(dataset2_path, augmented2_path)

Processing cry: 100%|██████████| 441/441 [01:07<00:00,  6.57it/s]
Processing donateacry_corpus_cleaned_and_updated_data: 100%|██████████| 1271/1271 [07:10<00:00,  2.95it/s]


In [ ]:
def balance_datasets(path1, path2, final_path, labels_csv_path):
    files1 = get_all_audio_files(path1)
    files2 = get_all_audio_files(path2)

    if len(files1) == 0 and len(files2) == 0:
        print("⚠️ No audio files found in both datasets.")
        return
    if len(files1) == 0:
        print("⚠️ Dataset1 empty. Using Dataset2 only.")
        all_files = files2
        labels_data = [[os.path.basename(f), "dataset2"] for f in all_files]
    elif len(files2) == 0:
        print("⚠️ Dataset2 empty. Using Dataset1 only.")
        all_files = files1
        labels_data = [[os.path.basename(f), "dataset1"] for f in all_files]
    else:
        # Balance datasets
        len1, len2 = len(files1), len(files2)
        print(f"Dataset1: {len1} files, Dataset2: {len2} files")

        if len1 > len2:
            big, small = files1, files2
            big_label, small_label = "dataset1", "dataset2"
        else:
            big, small = files2, files1
            big_label, small_label = "dataset2", "dataset1"

        diff = abs(len(big) - len(small))
        print(f"Oversampling {small_label} by {diff} files...")
        small_oversampled = small + random.choices(small, k=diff)

        all_files = big + small_oversampled
        random.shuffle(all_files)

        labels_data = []
        for f in all_files:
            label = "dataset1" if f in files1 else "dataset2"
            labels_data.append([os.path.basename(f), label])

    # Copy to final folder
    os.makedirs(final_path, exist_ok=True)
    for f in tqdm(all_files, desc="Saving final balanced dataset"):
        try:
            shutil.copy(f, os.path.join(final_path, os.path.basename(f)))
        except Exception as e:
            print(f"⚠️ Skipped copying {f}: {e}")

    # Save CSV
    df = pd.DataFrame(labels_data, columns=["filename", "label"])
    df.to_csv(labels_csv_path, index=False)
    print(f"✅ Final balanced dataset saved at: {final_path}")
    print(f"✅ Labels CSV saved at: {labels_csv_path}")
    print(f"Total files: {len(all_files)}")

balance_datasets(augmented1_path, augmented2_path, final_path, labels_csv_path)

Dataset1: 1764 files, Dataset2: 5444 files
Oversampling dataset1 by 3680 files...


Saving final balanced dataset: 100%|██████████| 10888/10888 [00:12<00:00, 850.98it/s] 


✅ Final balanced dataset saved at: /content/Merged_Final
✅ Labels CSV saved at: /content/labels.csv
Total files: 10888


In [ ]:
!zip -r /content/Merged_Final.zip /content/Merged_Final /content/labels.csv

from google.colab import files
files.download("/content/Merged_Final.zip")